# Step 8 -- Mask-based Anomaly Segmentation Baselines (EoMT)

**Objective:** Extend the anomaly detection pipeline to a **mask-architecture**
model (EoMT with DINOv2 backbone). The model outputs **mask logits + class
logits** which are combined into per-pixel scores via:

$$S_{c,h,w} = \sum_{q} \; \sigma(\text{mask\_logits}_{q,h,w}) \;\cdot\;
\text{softmax}(\text{class\_logits}_q)_c$$

where $q$ indexes learnable queries, and $\sigma$ is the sigmoid function.

**Methods implemented:**
- **MSP, Max Logit, Max Entropy** (same as Step 7)
- **RbA** (Rejected by All): $-\sum_c \tanh(f_c)$ -- only meaningful with
  mask architectures

**Checkpoints evaluated:**
1. `eomt_coco.bin` -- pretrained on COCO panoptic
2. `eomt_cityscapes.bin` -- pretrained on Cityscapes semantic
3. `eomt_coco_full_finetuned_cityscapes.bin` -- COCO -> fine-tuned on Cityscapes

## 1. Imports & Path Configuration

In [ ]:
import glob
import json
import os
import os.path as osp
import sys
from collections import defaultdict
from typing import Tuple

# Ensure eomt package is importable
# (Assumes this notebook lives in eval/ alongside the eomt/ directory)
_NB_DIR = os.getcwd()
_ROOT = osp.abspath(osp.join(_NB_DIR, ".."))
_EOMT_DIR = osp.join(_ROOT, "eomt")
sys.path.insert(0, _EOMT_DIR)
sys.path.insert(0, _ROOT)

import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image
from torch import nn, optim
from torchvision.datasets import Cityscapes
from torchvision.transforms import Compose, Resize, ToTensor

from eomt.models.eomt import EoMT
from eomt.models.vit import ViT
from eomt.training.mask_classification_semantic import MaskClassificationSemantic

# --- Reproducibility ---
seed = 42
np.random.seed(seed)
torch.manual_seed(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# --- Checkpoint paths ---
CHECKPOINT_DIR = osp.join(_ROOT, "eomt", "checkpoints")
CHECKPOINTS = {
    "COCO":             osp.join(CHECKPOINT_DIR, "eomt_coco.bin"),
    "Cityscapes":       osp.join(CHECKPOINT_DIR, "eomt_cityscapes.bin"),
    "COCO->Cityscapes": osp.join(CHECKPOINT_DIR, "eomt_coco_full_finetuned_cityscapes.bin"),
}

DATASETS = {
    "RoadAnomaly21":  osp.join(_ROOT, "Validation_Dataset/RoadAnomaly21/images/*.png"),
    "RoadObsticle21": osp.join(_ROOT, "Validation_Dataset/RoadObsticle21/images/*.webp"),
    "FS_LostFound":   osp.join(_ROOT, "Validation_Dataset/FS_LostFound_full/images/*.png"),
    "FS_Static":      osp.join(_ROOT, "Validation_Dataset/fs_static/images/*.jpg"),
    "RoadAnomaly":    osp.join(_ROOT, "Validation_Dataset/RoadAnomaly/images/*.jpg"),
}

METHODS = ["msp", "max_logit", "max_entropy", "rba"]

print("Checkpoints found:")
for name, path in CHECKPOINTS.items():
    exists = "V" if osp.exists(path) else "X MISSING"
    print(f"  [{exists}]  {name}: {path}")


## 2. Evaluation Metrics & Anomaly Score Functions

Same AuPRC / FPR@95TPR implementations as Step 7, plus the **RbA** method
that leverages mask architecture outputs.

In [ ]:
# --- Metrics (same as Step 7) ---

def average_precision_score(y_true: np.ndarray, y_score: np.ndarray) -> float:
    y_true = np.asarray(y_true).astype(np.int64)
    y_score = np.asarray(y_score).astype(np.float64)
    if y_true.ndim != 1 or y_score.ndim != 1 or y_true.shape[0] != y_score.shape[0]:
        raise ValueError("y_true and y_score must be 1D arrays of same length")
    pos = int(np.sum(y_true == 1))
    if pos == 0:
        return 0.0
    order = np.argsort(-y_score, kind="mergesort")
    y_true_sorted = y_true[order]
    tp = np.cumsum(y_true_sorted == 1)
    fp = np.cumsum(y_true_sorted == 0)
    precision = tp / np.maximum(tp + fp, 1)
    recall = tp / pos
    distinct = np.r_[True, y_score[order][1:] != y_score[order][:-1]]
    precision = precision[distinct]
    recall = recall[distinct]
    recall = np.r_[0.0, recall]
    precision = np.r_[precision[0], precision]
    return float(np.sum((recall[1:] - recall[:-1]) * precision[1:]))


def fpr_at_95_tpr(y_score: np.ndarray, y_true: np.ndarray) -> float:
    y_true = np.asarray(y_true).astype(np.int64)
    y_score = np.asarray(y_score).astype(np.float64)
    if y_true.ndim != 1 or y_score.ndim != 1 or y_true.shape[0] != y_score.shape[0]:
        raise ValueError("y_true and y_score must be 1D arrays of same length")
    pos = int(np.sum(y_true == 1))
    neg = int(np.sum(y_true == 0))
    if pos == 0 or neg == 0:
        return 0.0
    order = np.argsort(-y_score, kind="mergesort")
    y_true_sorted = y_true[order]
    tp = np.cumsum(y_true_sorted == 1)
    fp = np.cumsum(y_true_sorted == 0)
    tpr = tp / pos
    fpr = fp / neg
    idx = np.where(tpr >= 0.95)[0]
    if idx.size == 0:
        return 1.0
    return float(np.min(fpr[idx]))


# --- Anomaly scores from PER-PIXEL logits (already combined) ---

def anomaly_map_from_scores(
    scores: torch.Tensor, method: str, temperature: float = 1.0
) -> np.ndarray:
    """Convert per-pixel class scores to an anomaly heatmap.

    Args:
        scores: Tensor of shape (C, H, W) -- per-pixel class logits.
        method: "msp" | "max_logit" | "max_entropy" | "rba"
        temperature: T for scaling (applied before softmax).

    Returns:
        np.ndarray of shape (H, W).
    """
    raw_scores = scores.float()
    temperature = max(float(temperature), 1e-8)
    scaled_scores = raw_scores / temperature
    probs = torch.softmax(scaled_scores, dim=0)

    if method == "msp":
        return (1.0 - probs.max(dim=0).values).detach().cpu().numpy()
    if method == "max_logit":
        return (-scaled_scores.max(dim=0).values).detach().cpu().numpy()
    if method == "max_entropy":
        log_probs = torch.log_softmax(scaled_scores, dim=0)
        entropy = -(probs * log_probs).sum(dim=0)
        return entropy.detach().cpu().numpy()
    if method == "rba":
        # RbA: sum of -tanh(logits) across classes
        return (-torch.tanh(raw_scores).sum(dim=0)).detach().cpu().numpy()
    raise ValueError(f"Unknown method: {method}")


def infer_dataset_name(input_pattern: str) -> str:
    norm = input_pattern.replace("\\", "/")
    parts = [p for p in norm.split("/") if p]
    if "Validation_Dataset" in parts:
        idx = parts.index("Validation_Dataset")
        if idx + 1 < len(parts):
            return parts[idx + 1]
    if len(parts) >= 2:
        return parts[-2]
    return input_pattern


print("Metrics and anomaly score functions ready.")


## 3. Ground-Truth Mask Loader

Same logic as Step 7 -- handles the idiosyncrasies of each dataset's
label format.

In [ ]:
def load_gt_mask(path: str, pred_hw: Tuple[int, int]) -> np.ndarray | None:
    """Load and normalise a ground-truth anomaly mask.

    Returns None if the mask file is missing; returns a (H, W) array
    where 0 = in-distribution and 1 = anomaly.
    """
    pathGT = path.replace("images", "labels_masks")
    if "RoadObsticle21" in pathGT:
        pathGT = pathGT.replace("webp", "png")
    if "fs_static" in pathGT:
        pathGT = pathGT.replace("jpg", "png")
    if "RoadAnomaly" in pathGT:
        pathGT = pathGT.replace("jpg", "png")

    if not osp.exists(pathGT):
        return None

    mask = Image.open(pathGT)
    if mask.size != (pred_hw[1], pred_hw[0]):
        mask = mask.resize((pred_hw[1], pred_hw[0]), Image.NEAREST)
    ood_gts = np.array(mask)

    if "RoadAnomaly" in pathGT:
        ood_gts = np.where((ood_gts == 2), 1, ood_gts)
    if ("LostAndFound" in pathGT) or ("LostFound" in pathGT) or ("FS_LostFound_full" in pathGT):
        unique_vals = set(np.unique(ood_gts).tolist())
        if not unique_vals.issubset({0, 1, 255}):
            ood_gts = np.where((ood_gts == 0), 255, ood_gts)
            ood_gts = np.where((ood_gts == 1), 0, ood_gts)
            ood_gts = np.where((ood_gts > 1) & (ood_gts < 201), 1, ood_gts)
    if "Streethazard" in pathGT:
        ood_gts = np.where((ood_gts == 14), 255, ood_gts)
        ood_gts = np.where((ood_gts < 20), 0, ood_gts)
        ood_gts = np.where((ood_gts == 255), 1, ood_gts)

    return ood_gts


print("Ground-truth loader ready.")


## 4. EoMT Model Builder

We build the model **from a checkpoint alone** by inferring hyperparameters
(`num_classes`, `num_q`, `num_blocks`) directly from the weight shapes.
This makes the notebook independent of any training config files.

### Architecture overview

```
Input Image
    |
    v
DINOv2 ViT (frozen backbone)
    |
    +-- Q-Embeddings (learnable queries) --> Class Head --> class_logits (B, Q, C+1)
    |
    +-- Patch Features --> Mask Head --> mask_logits (B, Q, H/4, W/4)
                                          |
                                          v
                              Upscale (ScaleBlocks)
                                          |
                                          v
                              mask_logits (B, Q, H, W)
```

The per-pixel scores are obtained by:
```
scores = einsum('bqhw, bqc -> bchw', mask_logits.sigmoid(), class_logits.softmax(dim=-1)[..., :-1])
```


In [ ]:
def load_checkpoint_state(ckpt_path: str) -> dict:
    """Load checkpoint, stripping Lightning wrapper if present."""
    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=True)
    if "state_dict" in ckpt:
        ckpt = ckpt["state_dict"]
    return {k: v for k, v in ckpt.items() if "criterion.empty_weight" not in k}


def infer_model_hparams_from_ckpt(ckpt: dict) -> dict:
    """Infer num_classes, num_q, num_blocks from checkpoint tensor shapes."""
    q_key = "network.q.weight"
    class_key = "network.class_head.weight"
    blocks_key = "network.attn_mask_probs"

    if q_key not in ckpt or class_key not in ckpt:
        raise KeyError(f"Checkpoint missing {q_key} or {class_key}")

    num_q = int(ckpt[q_key].shape[0])
    num_classes = int(ckpt[class_key].shape[0] - 1)  # +1 for "no object"
    num_blocks = int(ckpt[blocks_key].numel()) if blocks_key in ckpt else 3

    return {"num_q": num_q, "num_classes": num_classes, "num_blocks": num_blocks}


def build_model(
    ckpt_path: str,
    img_size: tuple = (1024, 1024),
    backbone_name: str = "vit_base_patch14_reg4_dinov2",
    masked_attn_enabled: bool = True,
    attn_mask_annealing_enabled: bool = False,
) -> MaskClassificationSemantic:
    """Build an EoMT MaskClassificationSemantic model from a checkpoint."""
    ckpt = load_checkpoint_state(ckpt_path)
    hparams = infer_model_hparams_from_ckpt(ckpt)
    print(f"  Inferred: num_classes={hparams['num_classes']}, "
          f"num_q={hparams['num_q']}, num_blocks={hparams['num_blocks']}")

    encoder = ViT(
        img_size=img_size, backbone_name=backbone_name, ckpt_path=ckpt_path
    )
    network = EoMT(
        encoder=encoder,
        num_classes=hparams["num_classes"],
        num_q=hparams["num_q"],
        num_blocks=hparams["num_blocks"],
        masked_attn_enabled=masked_attn_enabled,
    )
    model = MaskClassificationSemantic(
        network=network,
        img_size=img_size,
        num_classes=hparams["num_classes"],
        attn_mask_annealing_enabled=attn_mask_annealing_enabled,
        ckpt_path=None,
        load_ckpt_class_head=True,
    ).to(device).eval()

    # --- Handle positional embedding size mismatch ---
    pos_key = "network.encoder.backbone.pos_embed"
    if pos_key in ckpt:
        target_grid = model.network.encoder.backbone.patch_embed.grid_size
        target_n = int(target_grid[0] * target_grid[1])
        if ckpt[pos_key].shape[1] != target_n:
            pos = ckpt[pos_key]
            old_n = int(pos.shape[1])
            old_h = int(round(old_n ** 0.5))
            old_w = old_h
            pos = pos.reshape(1, old_h, old_w, -1).permute(0, 3, 1, 2)
            pos = F.interpolate(
                pos, size=target_grid, mode="bicubic", align_corners=False
            )
            pos = pos.permute(0, 2, 3, 1).reshape(1, target_n, -1)
            ckpt[pos_key] = pos
            print("  Interpolated pos_embed to match target grid.")

    model.load_state_dict(ckpt, strict=False)
    print("  Model built and weights loaded.")
    return model


print("Model builder ready.")


## 5. EoMT Inference Pipeline

This is the core difference from Step 7. The mask architecture output
requires conversion from **(mask_logits, class_logits)** to
**per-pixel class scores**:

$$\text{scores}_{c,h,w} = \sum_q \;
\underbrace{\sigma(\text{mask\_logits}_{q,h,w})}_{\text{spatial mask}} \;\cdot\;
\underbrace{\text{softmax}(\text{class\_logits}_q)_c}_{\text{class distribution}}$$

The model also handles windowing (tiling) for images larger than `img_size`.

In [ ]:
@torch.no_grad()
def infer_scores_semantic(
    model: MaskClassificationSemantic, img_uint8_chw: torch.Tensor
) -> torch.Tensor:
    """Run EoMT inference and return per-pixel class logits.

    Args:
        model: MaskClassificationSemantic instance.
        img_uint8_chw: uint8 image tensor of shape (C, H, W).

    Returns:
        Per-pixel logits of shape (num_classes, H, W).
    """
    imgs = [img_uint8_chw.to(next(model.parameters()).device)]
    img_sizes = [img_uint8_chw.shape[-2:]]

    # Window into crops (model.img_size) if needed
    crops, origins = model.window_imgs_semantic(imgs)

    # Forward pass -> mask_logits + class_logits per transformer block
    mask_logits_per_layer, class_logits_per_layer = model(crops)

    # Use the final layer outputs
    mask_logits = mask_logits_per_layer[-1]
    class_logits = class_logits_per_layer[-1]

    # Upsample mask logits to model.img_size
    mask_logits = F.interpolate(mask_logits, model.img_size, mode="bilinear")

    # Combine mask x class -> per-pixel logits
    #   einsum('bqhw, bqc -> bchw', mask_logits.sigmoid(), class_logits.softmax(dim=-1)[..., :-1])
    crop_scores = model.to_per_pixel_logits_semantic(mask_logits, class_logits)

    # Stitch crops back & resize to original image size
    scores_list = model.revert_window_logits_semantic(crop_scores, origins, img_sizes)
    return scores_list[0]   # (C, H, W)


print("EoMT inference pipeline ready.")


## 6. Run Evaluation -- All Checkpoints x All Datasets x All Methods

This is the main evaluation loop. For efficiency, each image is passed through
the model **once**, and all four anomaly scoring methods are applied to the
same per-pixel scores.

> **Note:** This section runs 3 checkpoints x 5 datasets = 15 forward passes
> over the full validation sets. On a GPU this takes ~30-60 minutes per
> checkpoint. On CPU it may take several hours. The `torch.cuda.empty_cache()`
> calls help manage GPU memory between checkpoints.

In [ ]:
# --- Master evaluation loop ---
# all_results[ckpt_name][ds_name][method] = {"auprc": ..., "fpr95": ...}

all_results = {}

for ckpt_name, ckpt_path in CHECKPOINTS.items():
    if not osp.exists(ckpt_path):
        print(f"\nSKIP {ckpt_name}: checkpoint not found at {ckpt_path}")
        continue

    print(f"\n{'#'*70}")
    print(f"# Checkpoint: {ckpt_name}")
    print(f"# Path: {ckpt_path}")
    print(f"{'#'*70}")

    # --- Build model once per checkpoint ---
    model = build_model(ckpt_path)
    all_results[ckpt_name] = {}

    for ds_name, ds_pattern in DATASETS.items():
        image_paths = sorted(glob.glob(os.path.expanduser(ds_pattern)))
        print(f"\n  Dataset: {ds_name}  ({len(image_paths)} images)")

        if len(image_paths) == 0:
            print("    WARNING: No images found -- skipping.")
            continue

        all_results[ckpt_name][ds_name] = {}

        # --- Collect per-pixel scores for all images ---
        all_scores = []   # list of (C, H, W) tensors on CPU
        all_gts = []      # list of (H, W) np arrays

        for path in image_paths:
            img = Image.open(path).convert("RGB")
            img_np = np.array(img)
            img_uint8 = torch.from_numpy(img_np).permute(2, 0, 1).contiguous()

            scores = infer_scores_semantic(model, img_uint8)   # (C, H, W)
            gt = load_gt_mask(path, pred_hw=scores.shape[-2:])

            if gt is None:
                continue
            if 1 not in np.unique(gt):
                continue

            all_scores.append(scores.cpu())
            all_gts.append(gt)

        if len(all_gts) == 0:
            print("    No valid samples -- skipping.")
            continue

        # --- Evaluate each method on the SAME scores ---
        for method in METHODS:
            anomaly_maps = []
            for sc in all_scores:
                amap = anomaly_map_from_scores(sc, method, temperature=1.0)
                anomaly_maps.append(amap)

            gt_all = np.concatenate([g.flatten() for g in all_gts])
            scores_all = np.concatenate([a.flatten() for a in anomaly_maps])

            ood_mask = (gt_all == 1)
            ind_mask = (gt_all == 0)

            ood_out = scores_all[ood_mask]
            ind_out = scores_all[ind_mask]

            val_out = np.concatenate([ind_out, ood_out])
            val_label = np.concatenate([np.zeros(len(ind_out)), np.ones(len(ood_out))])

            auprc = average_precision_score(val_label, val_out)
            fpr95 = fpr_at_95_tpr(val_out, val_label)

            all_results[ckpt_name][ds_name][method] = {"auprc": auprc, "fpr95": fpr95}
            print(f"    {method:>12s}  |  AuPRC: {auprc*100:5.2f}%  "
                  f"|  FPR@95: {fpr95*100:5.2f}%")

    # Free model memory before loading next checkpoint
    del model
    torch.cuda.empty_cache()

print("\n\nAll evaluations complete!")


## 7. Temperature Scaling for EoMT

Same procedure as Step 7: collect logits on the Cityscapes validation set,
then fit a single temperature $T$ by minimising NLL with LBFGS.

This is done **per checkpoint** since each set of weights will have its own
optimal temperature.

In [ ]:
# --- Import helpers from Step 7 (reuse LabelIdsToTrainIds, SegmentationTemperatureScaler) ---
# Make sure eval/ is on the path
sys.path.insert(0, osp.join(_ROOT, "eval"))

from dataset import cityscapes
from fit_temperature_erfnet import (
    IGNORE_INDEX, LabelIdsToTrainIds, SegmentationTemperatureScaler
)

CITYSCAPES_DIR = osp.join(_ROOT, "Cityscapes val")

if not osp.exists(CITYSCAPES_DIR):
    print(f"WARNING: Cityscapes directory not found at '{CITYSCAPES_DIR}'.")
    print("Temperature scaling requires the Cityscapes validation set.")
    print("Please download it and update CITYSCAPES_DIR.")
else:
    input_transform_cs = Compose([Resize((512, 1024), Image.BILINEAR), ToTensor()])
    target_transform_cs = Compose([
        Resize((512, 1024), Image.NEAREST), LabelIdsToTrainIds()
    ])

    # --- Fit temperature for EACH checkpoint ---
    temperature_results = {}

    for ckpt_name, ckpt_path in CHECKPOINTS.items():
        if not osp.exists(ckpt_path):
            continue
        print(f"\nFitting temperature for: {ckpt_name}")

        # Use smaller img_size for faster temperature fitting
        model = build_model(ckpt_path, img_size=(640, 640))
        model_device = next(model.parameters()).device

        loader = torch.utils.data.DataLoader(
            cityscapes(CITYSCAPES_DIR, input_transform_cs, target_transform_cs,
                       subset="val", label_suffix="_labelIds.png"),
            num_workers=4, batch_size=1, shuffle=False,
        )

        logits_list, labels_list = [], []
        MAX_PIXELS = 4096

        with torch.no_grad():
            for step, (images, labels, filename, _) in enumerate(loader):
                image_uint8 = (images[0] * 255.0).round().clamp(0, 255).to(torch.uint8)
                image_uint8 = image_uint8.to(model_device)
                scores = infer_scores_semantic(model, image_uint8)

                # Flatten and filter ignore pixels
                logits_flat = scores.permute(1, 2, 0).reshape(-1, scores.shape[0])
                labels_flat = labels.squeeze(0).reshape(-1)
                valid = labels_flat != IGNORE_INDEX
                logits_flat = logits_flat[valid]
                labels_flat = labels_flat[valid]

                if logits_flat.numel() == 0:
                    continue
                if logits_flat.shape[0] > MAX_PIXELS:
                    idx = torch.randperm(logits_flat.shape[0])[:MAX_PIXELS]
                    logits_flat = logits_flat[idx]
                    labels_flat = labels_flat[idx]

                logits_list.append(logits_flat.cpu())
                labels_list.append(labels_flat.cpu())

                if step % 25 == 0:
                    print(f"  [{step}] {logits_flat.shape[0]} pixels "
                          f"from {osp.basename(filename[0])}")

        logits_all = torch.cat(logits_list, dim=0)
        labels_all = torch.cat(labels_list, dim=0)
        print(f"  Collected: logits {tuple(logits_all.shape)}, "
              f"labels {tuple(labels_all.shape)}")

        scaler = SegmentationTemperatureScaler(init_temperature=1.5).to(model_device)
        stats = scaler.set_temperature(logits_all, labels_all, model_device)
        temperature_results[ckpt_name] = stats["temperature"]
        print(f"  Best T for {ckpt_name}: {stats['temperature']:.4f}")

        del model
        torch.cuda.empty_cache()

    print("\nTemperature fitting complete:")
    for name, t in temperature_results.items():
        print(f"  {name}: T = {t:.4f}")


## 8. Results Summary Table

Compile all results into the assignment table format:

| Model | mIoU | Method | SMIYC RA-21 AuPRC | FPR95 | SMIYC RO-21 AuPRC | FPR95 | FS L&F AuPRC | FPR95 | FS Static AuPRC | FPR95 | Road Anomaly AuPRC | FPR95 |
|-------|------|--------|-------------------|-------|-------------------|-------|-------------|-------|---------------|-------|--------------------|-------|
| EoMT (COCO) | | MSP | | | | | | | | | | |
| EoMT (COCO) | | Max Logit | | | | | | | | | | |
| ... | | | | | | | | | | | | |

For temperature scaling, add rows like MSP (T=0.5), MSP (T=0.75), MSP (best T).

In [ ]:
print("=" * 80)
print("FINAL RESULTS -- Step 8 Mask-based Baselines")
print("=" * 80)

for ckpt_name in all_results:
    print(f"\n{'--'*35}")
    print(f"Checkpoint: {ckpt_name}")
    print(f"{'--'*35}")
    print(f"{'Method':>12s} | {'Dataset':<20s} | {'AuPRC':>8s} | {'FPR@95':>8s}")
    print(f"{'-'*12}-+-{'-'*20}-+-{'-'*8}-+-{'-'*8}")

    for ds_name in DATASETS:
        if ds_name not in all_results[ckpt_name]:
            continue
        for method in METHODS:
            if method not in all_results[ckpt_name][ds_name]:
                continue
            r = all_results[ckpt_name][ds_name][method]
            print(f"{method:>12s} | {ds_name:<20s} "
                  f"| {r['auprc']*100:7.2f}% | {r['fpr95']*100:7.2f}%")

# --- Optional: pandas pivot table ---
try:
    import pandas as pd
    rows = []
    for ckpt_name in all_results:
        for ds_name in all_results[ckpt_name]:
            for method in all_results[ckpt_name][ds_name]:
                r = all_results[ckpt_name][ds_name][method]
                rows.append({
                    "Model": "EoMT", "Checkpoint": ckpt_name,
                    "Method": method, "Dataset": ds_name,
                    "AuPRC": round(r["auprc"] * 100, 2),
                    "FPR95": round(r["fpr95"] * 100, 2),
                })
    df = pd.DataFrame(rows)
    pivot = df.pivot_table(
        index=["Model", "Checkpoint", "Method"],
        columns="Dataset",
        values=["AuPRC", "FPR95"],
        aggfunc="first"
    )
    print("\n\n=== Assignment-ready pivot table ===")
    print(pivot.to_string())
except ImportError:
    print("\n(pandas not available -- 'pip install pandas' for pivot table)")


## 9. Temperature Scaling Results

To complete the temperature scaling table from the assignment, re-run the
evaluation loop with `temperature=<value>` in `anomaly_map_from_scores()`:

```python
TEST_TEMPERATURES = [0.5, 0.75, 1.0, 1.1]  # + best_t from fitting

for ckpt_name in CHECKPOINTS:
    model = build_model(CHECKPOINTS[ckpt_name])
    for ds_name, ds_pattern in DATASETS.items():
        for T in TEST_TEMPERATURES:
            # ... same loop as Section 6, but with temperature=T
            amap = anomaly_map_from_scores(scores, "msp", temperature=T)
    del model
```

| Method | mIoU | SMIYC RA-21 AuPRC | FPR95 | ... |
|--------|------|-------------------|-------|-----|
| MSP | | | | |
| MSP (T=0.5) | | | | |
| MSP (T=0.75) | | | | |
| MSP (T=1.1) | | | | |
| MSP (best T) | | | | |

> Use the fitted `temperature_results` dict from Section 7 to fill in
> `best_t` for each checkpoint.

**End of Step 8 notebook.**